In [1]:
# pip install -q crewai langchain-groq langchain-community langchain-text-splitters faiss-cpu deepeval datasets python-dotenv

In [2]:
import os
from dotenv import load_dotenv
load_dotenv() 
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["OPENAI_API_KEY"] = GROQ_API_KEY
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"
print(" GROQ API key set.")

 GROQ API key set.


In [3]:
KNOWLEDGE_BASE = """
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Machine Learning (ML) is a subset of AI that enables systems to learn from data.

Deep Learning uses neural networks with multiple layers.
Transformers revolutionized NLP using attention mechanisms.

RAG (Retrieval-Augmented Generation) retrieves external documents to improve responses.
It reduces hallucinations by grounding answers in real data.

Fine-tuning adapts a model to a specific task.
LoRA is an efficient fine-tuning method using low-rank matrices.

Prompt engineering improves outputs without retraining.
"""

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = splitter.create_documents([KNOWLEDGE_BASE])

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector store ready.")

C:\Users\Meghana Veeramallu\AppData\Local\Temp\ipykernel_14456\3811191413.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Vector store ready.


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # safer for rate limits
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

In [6]:
def rag_tool(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])

    prompt = f"""
    Answer using ONLY this context:
    {context}

    Question: {question}
    """

    response = llm.invoke(prompt).content

    return {
        "answer": response,
        "context": context
    }

print("RAG tool ready.")

RAG tool ready.


In [7]:
from deepeval.models import DeepEvalBaseLLM
from langchain_groq import ChatGroq
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

class GroqJudge(DeepEvalBaseLLM):
    def __init__(self):
        self.model = ChatGroq(
            model="llama-3.1-8b-instant",  # safer for rate limits
            temperature=0,
            groq_api_key=GROQ_API_KEY
        )

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        return self.model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        res = await self.model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return "groq/llama-3.1-8b-instant"

judge = GroqJudge()

faithfulness = FaithfulnessMetric(threshold=0.7, model=judge)
relevancy = AnswerRelevancyMetric(threshold=0.7, model=judge)

print("Evaluator ready.")

Evaluator ready.


In [8]:
from deepeval.test_case import LLMTestCase

def evaluate_answer(question, answer, context):
    tc = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[context]
    )

    faithfulness.measure(tc)
    relevancy.measure(tc)

    return {
        "faithfulness": faithfulness.score,
        "relevancy": relevancy.score,
        "pass": faithfulness.is_successful() and relevancy.is_successful(),
        "reason": f"{faithfulness.reason} | {relevancy.reason}"
    }

In [9]:
def revise_answer(question, answer, context, reason):
    prompt = f"""
    Improve the answer using context.

    Question: {question}
    Context: {context}

    Original Answer: {answer}
    Issues: {reason}

    Provide a corrected answer.
    """

    return llm.invoke(prompt).content

In [10]:
import time

def safe_call(func, *args):
    while True:
        try:
            return func(*args)
        except Exception as e:
            if "rate" in str(e).lower():
                print(" Waiting (rate limit)...")
                time.sleep(3)
            else:
                raise e


questions = [
    "What is RAG?",
    "Explain transformers",
    "What is LoRA?",
    "What is AI?",
    "Capital of France?"  # adversarial
]

results = []

for q in questions:
    print(f"\nQ: {q}")

    rag_out = safe_call(rag_tool, q)

    eval_out = safe_call(
        evaluate_answer,
        q,
        rag_out["answer"],
        rag_out["context"]
    )

    final_answer = rag_out["answer"]

    if not eval_out["pass"]:
        print("Failed — Revising...")
        final_answer = safe_call(
            revise_answer,
            q,
            rag_out["answer"],
            rag_out["context"],
            eval_out["reason"]
        )

    print("Answer:", final_answer[:200])

    results.append({
        "question": q,
        "faithfulness": eval_out["faithfulness"],
        "relevancy": eval_out["relevancy"],
        "passed": eval_out["pass"]
    })

    time.sleep(1)


Q: What is RAG?


Output()

Output()

Answer: RAG (Retrieval-Augmented Generation) is a model that retrieves external documents to improve responses.

Q: Explain transformers


Output()

...

Output()

 Waiting (rate limit)...


Output()

Output()

Failed — Revising...
Answer: The context provided is about Deep Learning, Transformers, fine-tuning, LoRA, prompt engineering, Artificial Intelligence, and Machine Learning, but it doesn't seem to be related to the capital of Fra


In [11]:
import pandas as pd

df = pd.DataFrame(results)
df

,question,faithfulness,relevancy,passed
0,What is RAG?,1.000000,1.000000,True
1,Explain transformers,0.857143,0.857143,True
2,What is LoRA?,1.000000,1.000000,True
3,What is AI?,1.000000,1.000000,True
4,Capital of France?,1.000000,0.000000,False
